In [2]:
import networkx as nx
import pandas as pd

# Load STRING network
string_data = pd.read_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/metadata/STRING_hs_interactions.txt", sep="\t", names=['gene1','gene2','weight'],header=0)
string_data

,gene1,gene2,weight
0,ARF5,DYRK4,0.166000
1,ARF5,PPP5C,0.254968
2,ARF5,MAP4K5,0.157276
3,ARF5,RALBP1,0.156000
4,ARF5,PKP2,0.160210
...,...,...,...
1972242,ZNF788,ANKRD65,0.162811
1972243,ZNF788,PCSK5,0.215378
1972244,ZNF788,OBSCN,0.236000
1972245,ZNF788,RAB5C,0.195550


In [3]:
# Create graph (assuming 'gene1', 'gene2', 'weight' columns)
G = nx.Graph()
for _, row in string_data.iterrows():
    G.add_edge(row['gene1'], row['gene2'], weight=1-row['weight'])


In [7]:
node_types = pd.read_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/node_types.tsv",sep="\t",names=['node','type'],header=0)
# Assuming the 'type' column contains labels like 'node' and 'target'
sources = node_types[node_types['type'] == 'source']['node'].tolist()
targets = node_types[node_types['type'] == 'target']['node'].tolist()

from joblib import Parallel, delayed

def process_source(source):
    paths = {}
    if source in G:
        paths_from_source = nx.single_source_dijkstra_path(G, source, weight='weight')
        for target in targets:
            paths[(source, target)] = paths_from_source.get(target, None)
    return paths

results = Parallel(n_jobs=-1)(delayed(process_source)(src) for src in sources)
shortest_paths = {k: v for d in results for k, v in d.items()}

In [8]:
shortest_paths

{('PARP1', 'EIF4E'): ['PARP1', 'UBC', 'AKT1', 'MTOR', 'EIF4EBP1', 'EIF4E'],
 ('PARP1', 'FKBP4'): ['PARP1', 'UBC', 'HSP90AA1', 'FKBP4'],
 ('PARP1', 'RBL2'): ['PARP1', 'UBC', 'E2F1', 'TFDP1', 'RBL1', 'E2F4', 'RBL2'],
 ('PARP1', 'FKBP1A'): ['PARP1', 'UBC', 'AKT1', 'MTOR', 'FKBP1A'],
 ('PARP1', 'SCN5A'): ['PARP1', 'UBC', 'CALM2', 'SCN5A'],
 ('PARP1', 'SCAP'): ['PARP1', 'UBC', 'HMGCR', 'INSIG1', 'SCAP'],
 ('PARP1', 'FOXA3'): ['PARP1',
  'UBC',
  'CUL1',
  'BTRC',
  'HIVEP1',
  'IRF2BP2',
  'FOXA3'],
 ('PARP1', 'COQ2'): ['PARP1', 'UBC', 'COQ2'],
 ('PARP1', 'HNF1A'): ['PARP1', 'UBC', 'CTNNB1', 'HNF1A'],
 ('PARP1', 'USP24'): ['PARP1', 'UBC', 'USP24'],
 ('PARP1', 'LPA'): ['PARP1', 'UBC', 'CANX', 'LPA'],
 ('PARP1', 'ACTR2'): ['PARP1', 'UBC', 'ACTR2'],
 ('PARP1', 'PAPOLA'): ['PARP1',
  'UBC',
  'POLR2A',
  'GTF2F1',
  'GTF2B',
  'TBP',
  'BRF1',
  'GTF3C3',
  'CPSF2',
  'CPSF3',
  'PAPOLA'],
 ('PARP1', 'SLIT1'): ['PARP1', 'UBC', 'USP33', 'ROBO1', 'SLIT1'],
 ('PARP1', 'TBP'): ['PARP1', 'UBC', 'POL

In [9]:
from collections import Counter

# Flatten the shortest paths
path_nodes = [node for path in shortest_paths.values() if path for node in path]
node_participation = Counter(path_nodes)

In [10]:
# Create a subgraph based on relevant shortest paths
relevant_edges = [(u, v) for path in shortest_paths.values() if path for u, v in zip(path, path[1:])]
subgraph = G.edge_subgraph(relevant_edges)

# Compute betweenness centrality
bc_scores = nx.betweenness_centrality(subgraph, weight='weight')

In [11]:
sorted_bc = sorted(bc_scores.items(), key=lambda x: x[1], reverse=True)

In [12]:
# Define BC cutoff (example: top 2.5%)
top_nodes_cutoff = int(len(sorted_bc) * 0.025)
important_nodes = [node for node, bc in sorted_bc[:top_nodes_cutoff]]

In [13]:
len(important_nodes)
important_nodes

['UBC',
 'CBL',
 'HSP90AA1',
 'RPS3',
 'ATP5A1',
 'ATP5B',
 'ESR1',
 'HSPA8',
 'CCNB1',
 'BCAS2',
 'POLR2A']

In [16]:
important_nodes_ = set(important_nodes) - set(sources) - set(targets)
print(type(shortest_paths))
# Subset dictionary
filtered_paths = {
    key: path for key, path in shortest_paths.items()
    if path is not None and any(gene in path for gene in important_nodes_)
}


<class 'dict'>


In [17]:
print(len(shortest_paths))
print(len(filtered_paths))
filtered_paths

3230
2882


{('PARP1', 'EIF4E'): ['PARP1', 'UBC', 'AKT1', 'MTOR', 'EIF4EBP1', 'EIF4E'],
 ('PARP1', 'FKBP4'): ['PARP1', 'UBC', 'HSP90AA1', 'FKBP4'],
 ('PARP1', 'RBL2'): ['PARP1', 'UBC', 'E2F1', 'TFDP1', 'RBL1', 'E2F4', 'RBL2'],
 ('PARP1', 'FKBP1A'): ['PARP1', 'UBC', 'AKT1', 'MTOR', 'FKBP1A'],
 ('PARP1', 'SCN5A'): ['PARP1', 'UBC', 'CALM2', 'SCN5A'],
 ('PARP1', 'SCAP'): ['PARP1', 'UBC', 'HMGCR', 'INSIG1', 'SCAP'],
 ('PARP1', 'FOXA3'): ['PARP1',
  'UBC',
  'CUL1',
  'BTRC',
  'HIVEP1',
  'IRF2BP2',
  'FOXA3'],
 ('PARP1', 'COQ2'): ['PARP1', 'UBC', 'COQ2'],
 ('PARP1', 'HNF1A'): ['PARP1', 'UBC', 'CTNNB1', 'HNF1A'],
 ('PARP1', 'USP24'): ['PARP1', 'UBC', 'USP24'],
 ('PARP1', 'LPA'): ['PARP1', 'UBC', 'CANX', 'LPA'],
 ('PARP1', 'ACTR2'): ['PARP1', 'UBC', 'ACTR2'],
 ('PARP1', 'PAPOLA'): ['PARP1',
  'UBC',
  'POLR2A',
  'GTF2F1',
  'GTF2B',
  'TBP',
  'BRF1',
  'GTF3C3',
  'CPSF2',
  'CPSF3',
  'PAPOLA'],
 ('PARP1', 'SLIT1'): ['PARP1', 'UBC', 'USP33', 'ROBO1', 'SLIT1'],
 ('PARP1', 'TBP'): ['PARP1', 'UBC', 'POL

In [18]:
# Convert to DataFrame
# Convert to DataFrame
filtered_paths_df = pd.DataFrame([
    {"Source": key[0], "Target": key[1], "Path": ",".join(path)}
    for key, path in filtered_paths.items()
])


filtered_paths_df
# Write DataFrame to CSV
filtered_paths_df.to_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/filtered_shortest_paths_bcscores.csv", index=False,sep='\t')

In [19]:
# Save results
pd.DataFrame(sorted_bc, columns=['Node', 'BC_Score']).to_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/bc_scores.csv", index=False)

In [21]:
import pandas as pd

# Load STRING network
# Assuming STRING network has 'gene1', 'gene2', and 'weight' columns
string_data = pd.read_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/metadata/STRING_hs_interactions.txt", sep="\t", names=['gene1','gene2','weight'],header=0)

# Extract all unique genes from the shortest paths
genes_in_paths = set(
    gene for path in filtered_paths.values() if path for gene in path
)

# Subset the STRING network by filtering rows where both genes are in `genes_in_paths`
subset_network = string_data[
    (string_data['gene1'].isin(genes_in_paths)) &
    (string_data['gene2'].isin(genes_in_paths))
]

# Save the subsetted network to a CSV
subset_network.to_csv("/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/subset_string_network_shortestpaths_bcscores.csv", index=False)

# Print a summary
print(f"Original STRING network size: {string_data.shape[0]} edges")
print(f"Subset STRING network size: {subset_network.shape[0]} edges")


Original STRING network size: 1972247 edges
Subset STRING network size: 10430 edges


In [22]:
# Compute average BC scores for each path
path_avg_bc = {}
for key, path in shortest_paths.items():
    if path:  # Ensure the path is not None
        avg_bc = sum(bc_scores.get(node, 0) for node in path) / len(path)
        path_avg_bc[key] = avg_bc

# Group paths by source node
paths_by_source = {}
for (source, target), avg_bc in path_avg_bc.items():
    if source not in paths_by_source:
        paths_by_source[source] = []
    paths_by_source[source].append(((source, target), avg_bc))

# Get top paths for each source node
top_percentage = 0.02  # Adjust this as needed
top_paths_per_source = {}

for source, paths in paths_by_source.items():
    # Sort paths by average BC score for the current source
    sorted_paths = sorted(paths, key=lambda x: x[1], reverse=True)
    num_top_paths = max(1, int(len(sorted_paths) * top_percentage))  # At least 1 path
    top_paths = sorted_paths[:num_top_paths]

    # Filter paths to include only those passing through important nodes
    filtered_paths = {
        key: shortest_paths[key]
        for key, avg_bc in top_paths
        if any(gene in shortest_paths[key] for gene in important_nodes_)
    }

    top_paths_per_source[source] = filtered_paths

# Print results for each source node
for source, paths in top_paths_per_source.items():
    print(f"Source: {source}")
    print(f"Number of top paths: {len(paths)}")
    for key, path in paths.items():
        print(f"  Target: {key[1]}, Path: {path}")
    print("-" * 30)



Source: PARP1
Number of top paths: 3
  Target: HMGCR, Path: ['PARP1', 'UBC', 'HMGCR']
  Target: HIF1A, Path: ['PARP1', 'UBC', 'HIF1A']
  Target: APOB, Path: ['PARP1', 'UBC', 'APOB']
------------------------------
Source: ADRB2
Number of top paths: 3
  Target: HMGCR, Path: ['ADRB2', 'UBC', 'HMGCR']
  Target: HIF1A, Path: ['ADRB2', 'UBC', 'HIF1A']
  Target: APOB, Path: ['ADRB2', 'UBC', 'APOB']
------------------------------
Source: ALAS1
Number of top paths: 3
  Target: HMGCR, Path: ['ALAS1', 'UBC', 'HMGCR']
  Target: HIF1A, Path: ['ALAS1', 'UBC', 'HIF1A']
  Target: APOB, Path: ['ALAS1', 'UBC', 'APOB']
------------------------------
Source: BIRC2
Number of top paths: 3
  Target: HMGCR, Path: ['BIRC2', 'UBC', 'HMGCR']
  Target: HIF1A, Path: ['BIRC2', 'UBC', 'HIF1A']
  Target: APOB, Path: ['BIRC2', 'UBC', 'APOB']
------------------------------
Source: BMP4
Number of top paths: 3
  Target: HMGCR, Path: ['BMP4', 'BMPR1A', 'UBC', 'HMGCR']
  Target: HIF1A, Path: ['BMP4', 'BMPR1A', 'UBC', 'HIF1

In [24]:
# Prepare data for DataFrame
csv_rows = []
for source, paths in top_paths_per_source.items():
    for (source_node, target_node), path in paths.items():
        if path:  # Ensure the path is not None
            # Calculate average BC score
            avg_bc = sum(bc_scores.get(node, 0) for node in path) / len(path)
            # Add row to list
            csv_rows.append({
                "Source": source_node,
                "Target": target_node,
                "Path": " -> ".join(path),  # Convert path list to a string
                "Average_BC_Score": avg_bc
            })

# Convert to DataFrame
filtered_paths_df = pd.DataFrame(csv_rows)

# Write DataFrame to CSV
output_file = "/projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/filtered_shortest_paths_bcscores_persource.csv"
filtered_paths_df.to_csv(output_file, index=False, sep='\t')

# Print confirmation
print(f"Top paths per source written to: {output_file}")


Top paths per source written to: /projects/ksamart@xsede.org/integrative-drugrep-tb/data/v2/figure3/filtered_shortest_paths_bcscores_persource.csv


In [29]:
# Compute degree centrality for all nodes
degree_centrality = nx.degree_centrality(G)

# Define threshold for hubs (top 1% or 2%, adjust as needed)
threshold = sorted(degree_centrality.values(), reverse=True)[int(len(degree_centrality) * 0.02)]

# Identify hub genes
hub_genes = {node for node, centrality in degree_centrality.items() if centrality >= threshold}

print(f"Number of hub genes detected: {len(hub_genes)}")
print(hub_genes)

Number of hub genes detected: 337
{'POLR2A', 'HDAC1', 'NOP56', 'TRIB1', 'NRIP3', 'STK4', 'GUCY2D', 'ADCY3', 'HSP90AB1', 'EHMT1', 'HIPK3', 'OASL', 'TRAP1', 'KRT76', 'STK40', 'PCSK6', 'PCSK5', 'PHLPP1', 'UBQLNL', 'IMP3', 'OAS1', 'BMP2K', 'CNBD2', 'RAD54L', 'CAPN13', 'SIN3B', 'RAD23B', 'TSSK2', 'PPP2CB', 'H2AFZ', 'ACTBL2', 'HSP90B1', 'PES1', 'PIM2', 'MAP4K5', 'ADCY5', 'MST4', 'NFYB', 'OAS3', 'YWHAH', 'ACTR10', 'TSSK4', 'TSSK6', 'ADCY8', 'SUMO2', 'LATS1', 'DYRK2', 'IRAK4', 'MAP4K3', 'IRAK2', 'MAPK4', 'CSNK2A1', 'PCSK1', 'DDX39A', 'LRRK2', 'YWHAG', 'LATS2', 'ZDHHC13', 'PDCD11', 'DHX8', 'POTEF', 'IMPDH2', 'PPP3R2', 'GMPS', 'PCSK4', 'PRKX', 'NPR2', 'CAPN3', 'ACTR1A', 'POLR1A', 'PCSK7', 'PITPNM2', 'POTEJ', 'UBBP4', 'ACTL7B', 'SIRT7', 'PRKACB', 'YWHAE', 'PHLPP2', 'STK38', 'XRN1', 'EEF1G', 'IRAK3', 'POLR2B', 'ADCY6', 'SUMO3', 'KDM2B', 'ACACA', 'UBA52', 'TCP1', 'ELAVL1', 'ANKDD1B', 'TAF1L', 'UMPS', 'CDK14', 'TSSK3', 'ACTRT1', 'AP001055.7', 'PNPLA6', 'POTEI', 'SUMO1', 'ACTA1', 'GNL3L', 'UBL4A', 'A

In [30]:
# Extract all unique genes (nodes) between sources and targets
intermediate_genes = set()

for source, paths in top_paths_per_source.items():
    for key, path in paths.items():
        if path:  # Ensure the path is not None
            # Exclude source and target nodes
            intermediate_genes.update(path[1:-1])

intermediate_genes = {gene for gene in intermediate_genes if gene not in hub_genes}

print(f"Number of unique intermediate genes: {len(intermediate_genes)}")
print("Genes between sources and targets:")
print(intermediate_genes)


Number of unique intermediate genes: 12
Genes between sources and targets:
{'PRPF8', 'INSR', 'CBX5', 'CD2', 'CANX', 'CD2BP2', 'TRIM28', 'BMPR1A', 'PRPF6', 'CCND2', 'CDKN1A', 'IRS1'}
